In [79]:
import os
from pyspark.sql.functions import col

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = "python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "python"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("AmazonReviews_Lab") \
    .master("local[1]") \
    .getOrCreate()

In [ ]:

rdd = spark.sparkContext.textFile("AmazonReviews.txt")
rdd.collect()

['product/productId: B000EVS4TY',
 'product/title: Arrowhead Mills Cookie Mix, Chocolate Chip, 12.9-Ounce Units (Pack of 6)',
 'product/price: unknown',
 'review/userId: A2SRVDDDOQ8QJL',
 'review/profileName: MJ23447',
 'review/helpfulness: 2/4',
 'review/score: 4.0',
 'review/time: 1206576000',
 'review/summary: Delicious cookie mix',
 "review/text: I thought it was funny that I bought this product without knowing it was a mix. I read the header very quickly and just thought it was packaged cookies. But no, it is cookie MIX and I guess I should have noticed that since it is right in the title.This is the first time I have ever tried baking with a cookie mix. If you are used to the convenience of the cookie dough that you buy wrapped up in plastic logs then you might be in for a bit of a surprise. Mixing up the dough can get VERY messy (it is extremely sticky). However, with a cookie mix like this you have a lot of flexibility in the ratio of ingredients (I like to add some extra butte

In [81]:
#create a list of diccionaries for every review

reviews_rdd = rdd.collect() #convert to a list

list_of_reviews = [] #analysed registers
review = {} #stores the processed review

for line in reviews_rdd:
    if line.strip() == "": #if the line is empty 
        if review: 
            list_of_reviews.append(review) #add review diccionary to list_of_reviews list
            review = {} #empty review for the new review
    else: #if the line is not empty
        key, value = line.split(": ", 1) #divides by the first ": " found
        review[key] = value #creates key and value inside the diccionary


In [82]:

df = spark.createDataFrame(list_of_reviews)
df.show(truncate=False)


+-------------+-----------------+------------------------------------------------------------------------+------------------+------------------------------------------------+------------+----------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [83]:

df1 = df.withColumn("review/score", col("review/score").cast("float"))
df2 = df1.groupBy("product/title").avg("review/score")
df2.show(5, False)

+-----------------------------------------------------------------------------------------------+-----------------+
|product/title                                                                                  |avg(review/score)|
+-----------------------------------------------------------------------------------------------+-----------------+
|3-10 oz. bottles: 2 Wilted Lettuce Salad Dressing, 1 Zest Sauce & Recipes                      |5.0              |
|4-10 oz. bottles: 2 Wilted Lettuce Dressing, 1 Zest Sauce, 1 Real McCoy Mustard Sauce & Recipes|5.0              |
|3-25 oz bottles: 2 Real McCoy Mustard Sauce, 1 Wilted Lettuce Dressing & Recipes               |5.0              |
|4-25 oz bottles: Real McCoy Mustard Sauce & Recipes                                            |5.0              |
|Milk Chocolate Brownie Pops with M&ms, 4 Pc Gift Box                                           |5.0              |
+-----------------------------------------------------------------------